# ECG Cardiovascular Disease Classification — DAT255 Project
**Authors:** Edvard Vindenes Steenslid & Morten Kvamme

**Objective:** Multi-label classification of cardiovascular conditions from the PTB-XL ECG dataset using 1D-CNN and LSTM architectures, with Grad-CAM interpretability.

**Metrics:** Accuracy, Recall, F1-Score (macro & per-class)

---

## Table of Contents
1. [Setup & Installations]
2. [Data Loading & Exploration]
3. [Preprocessing & Feature Engineering]
4. [Model 1 — 1D Convolutional Neural Network]
5. [Model 2 — LSTM Recurrent Network]
6. [Training with MLflow Tracking]
7. [Evaluation & Comparison]
8. [Grad-CAM Interpretability]
9. [Export for HuggingFace & Gradio Deployment]

## 1. Setup & Installations <a id='setup'></a>

In [ ]:
# Install required packages (run once)
# NOTE: Keras 3 is the standalone version — we import 'keras' directly, NOT 'tensorflow.keras'
!pip install -q keras tensorflow[and-cuda] wfdb pandas numpy matplotlib seaborn scikit-learn mlflow gradio huggingface_hub

In [ ]:
import os
# Remove any leftover SQLite tracking URI from the environment
os.environ.pop("MLFLOW_TRACKING_URI", None)
import ast
import zipfile
import numpy as np
import pandas as pd
import wfdb
import matplotlib.pyplot as plt
import seaborn as sns

# ── Keras 3 (standalone) ──────────────────────────────────────────────
# Keras 3 auto-detects the backend (TensorFlow, JAX, or PyTorch).
# We set TensorFlow explicitly for Grad-CAM compatibility later.
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers, callbacks, optimizers

# Scikit-learn metrics
from sklearn.metrics import (
    classification_report,
    f1_score,
    roc_auc_score,
    multilabel_confusion_matrix,
    accuracy_score,
)
from sklearn.preprocessing import MultiLabelBinarizer

print(f"Keras version : {keras.__version__}")
print(f"Keras backend : {keras.backend.backend()}")
print(f"NumPy version : {np.__version__}")

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────────
SEED = 42
keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

## 2. Data Loading & Exploration <a id='data'></a>

The **PTB-XL** dataset contains 21,799 twelve-lead ECG recordings of 10 seconds each, sampled at 100 Hz (low-res) and 500 Hz (high-res). Each record is annotated with up to five **diagnostic superclasses**:

| Superclass | Description |
|------------|-------------------------------|
| **NORM**   | Normal ECG |
| **MI**     | Myocardial Infarction |
| **STTC**   | ST/T Change |
| **CD**     | Conduction Disturbance |
| **HYP**    | Hypertrophy |

The dataset ships with a **10-fold stratified split** (`strat_fold` column). The standard convention is:
- Folds 1–8 → Training
- Fold 9 → Validation
- Fold 10 → Test

In [ ]:
# ── 2.1 Unzip the dataset ──────────────────────────────────────────
# Update this path to wherever your zip file is located
ZIP_PATH = "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip"
DATA_DIR = "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

if not os.path.isdir(DATA_DIR):
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(".")
    print("Done.")
else:
    print(f"Dataset directory already exists: {DATA_DIR}")

# List top-level contents
os.listdir(DATA_DIR)

In [ ]:
# ── 2.2  Load metadata ──────────────────────────────────────────────
df = pd.read_csv(os.path.join(DATA_DIR, "ptbxl_database.csv"), index_col="ecg_id")

# Parse the scp_codes column (stored as string repr of dict)
df["scp_codes"] = df["scp_codes"].apply(ast.literal_eval)

print(f"Total records: {len(df)}")
df.head()

In [ ]:
# ── 2.3 Clean metadata & map SCP codes to superclasses ─────────────
# Load the SCP statement lookup table
scp_df = pd.read_csv(os.path.join(DATA_DIR, "scp_statements.csv"), index_col=0)
scp_df = scp_df[scp_df["diagnostic"] == 1]  # keep only diagnostic codes

# Build a dict: scp_code -> diagnostic_class (superclass)
code_to_superclass = scp_df["diagnostic_class"].to_dict()

SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
NUM_CLASSES = len(SUPERCLASSES)

# ── Drop columns that are almost entirely NaN (identified in EDA) ──
# electrodes_problems (99.9%), infarction_stadium2 (99.5%),
# pacemaker (98.7%), burst_noise (97.2%), baseline_drift (92.7%),
# extra_beats (91.1%), static_noise (85.0%)
HIGH_NAN_COLS = [
    "electrodes_problems", "infarction_stadium2", "pacemaker",
    "burst_noise", "baseline_drift", "extra_beats", "static_noise",
]
df.drop(columns=[c for c in HIGH_NAN_COLS if c in df.columns], inplace=True)
print(f"Dropped {len(HIGH_NAN_COLS)} high-NaN columns")

# ── Remove age outliers (EDA found age=300) ────────────────────────
MAX_AGE = 120
n_age_outliers = (df["age"] > MAX_AGE).sum()
df = df[df["age"] <= MAX_AGE].copy()
print(f"Removed {n_age_outliers} records with age > {MAX_AGE}")

# Remove unnecessary columns
UNNECESSARY_COLS = ["patient_id","ecg_id", "filename_lr", "filename_hr",
                    "recording_date", "report"]

def extract_superclasses(scp_dict):
    """Given a dict of {scp_code: likelihood}, return the set of superclasses.
    
    Only codes with likelihood >= 50 are considered, matching the
    convention used in the PTB-XL benchmarking paper (Wagner et al., 2020).
    """
    classes = set()
    for code, likelihood in scp_dict.items():
        if code in code_to_superclass and likelihood >= 50:
            sc = code_to_superclass[code]
            if sc in SUPERCLASSES:
                classes.add(sc)
    return classes

df["superclasses"] = df["scp_codes"].apply(extract_superclasses)

# Drop records with no diagnostic superclass
n_before = len(df)
df = df[df["superclasses"].apply(len) > 0].copy()
print(f"\nRecords before filtering: {n_before}")
print(f"Records after filtering : {len(df)}")
print(f"Dropped (no diagnosis)  : {n_before - len(df)}")
print(f"Columns remaining       : {len(df.columns)}")

In [ ]:
# ── 2.4  Binarize labels (multi-label) ──────────────────────────────
mlb = MultiLabelBinarizer(classes=SUPERCLASSES)
labels = mlb.fit_transform(df["superclasses"])
df[SUPERCLASSES] = labels

print("Label distribution:")
for i, sc in enumerate(SUPERCLASSES):
    print(f"  {sc:5s}: {labels[:, i].sum():6d}  ({labels[:, i].mean()*100:.1f}%)")

print(f"\nMulti-label samples: {(labels.sum(axis=1) > 1).sum()} "
      f"({(labels.sum(axis=1) > 1).mean()*100:.1f}%)")

In [ ]:
# ── 2.5  Visualise label distribution ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of class counts
counts = [labels[:, i].sum() for i in range(NUM_CLASSES)]
axes[0].bar(SUPERCLASSES, counts, color="steelblue")
axes[0].set_title("Superclass Distribution")
axes[0].set_ylabel("Count")
for i, c in enumerate(counts):
    axes[0].text(i, c + 100, str(c), ha="center", fontsize=9)

# Number of labels per sample
n_labels = labels.sum(axis=1)
axes[1].hist(n_labels, bins=range(1, 7), align="left", color="coral", edgecolor="k")
axes[1].set_title("Labels per Sample")
axes[1].set_xlabel("Number of superclasses")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig("label_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 2.6 Load ECG waveforms ─────────────────────────────────────────
# We use the 100 Hz version (1000 timesteps x 12 leads) for speed.
# Switch to filename_hr for the 500 Hz version (5000 x 12) if desired.

SAMPLING_RATE = 100  # Hz

def load_raw_signals(df, data_dir, sampling_rate=100):
    """Load all ECG signals into a numpy array."""
    col = "filename_lr" if sampling_rate == 100 else "filename_hr"
    signals = []
    for idx, row in df.iterrows():
        fpath = os.path.join(data_dir, row[col])
        record = wfdb.rdrecord(fpath)
        signals.append(record.p_signal)  # shape: (timesteps, 12)
    return np.array(signals, dtype=np.float32)

print("Loading ECG signals (this takes a few minutes)...")
X_all = load_raw_signals(df, DATA_DIR, SAMPLING_RATE)
y_all = labels.astype(np.float32)

print(f"Signals shape : {X_all.shape}")   # (N, 1000, 12)
print(f"Labels shape  : {y_all.shape}")    # (N, 5)

In [ ]:
# ── 2.7  Visualise example ECG traces ───────────────────────────────
LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF",
              "V1", "V2", "V3", "V4", "V5", "V6"]

fig, axes = plt.subplots(4, 3, figsize=(16, 10), sharex=True)
sample_idx = 0
time = np.arange(X_all.shape[1]) / SAMPLING_RATE  # seconds

for i, ax in enumerate(axes.flat):
    ax.plot(time, X_all[sample_idx, :, i], linewidth=0.8, color="#2c3e50")
    ax.set_title(f"Lead {LEAD_NAMES[i]}", fontsize=10)
    ax.set_ylabel("mV")
    if i >= 9:
        ax.set_xlabel("Time (s)")

sc_labels = [SUPERCLASSES[j] for j in range(NUM_CLASSES) if y_all[sample_idx, j] == 1]
fig.suptitle(f"Sample {sample_idx} — Labels: {sc_labels}", fontsize=13)
plt.tight_layout()
plt.savefig("ecg_example.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Preprocessing & Feature Engineering <a id='preprocess'></a>

In [ ]:
# ── 3.1  Train / Validation / Test split (PTB-XL convention) ───────
folds = df["strat_fold"].values

train_mask = folds <= 8
val_mask   = folds == 9
test_mask  = folds == 10

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val,   y_val   = X_all[val_mask],   y_all[val_mask]
X_test,  y_test  = X_all[test_mask],  y_all[test_mask]

print(f"Train : {X_train.shape[0]:6d} samples")
print(f"Val   : {X_val.shape[0]:6d} samples")
print(f"Test  : {X_test.shape[0]:6d} samples")

In [ ]:
# ── 3.2  Z-score normalisation (per-channel, fitted on train) ──────
# This is standard practice for ECG — removes baseline wander effects
# and puts all leads on a comparable scale.

train_mean = X_train.mean(axis=(0, 1), keepdims=True)  # (1, 1, 12)
train_std  = X_train.std(axis=(0, 1), keepdims=True)   # (1, 1, 12)
train_std[train_std == 0] = 1.0  # avoid division by zero

X_train = (X_train - train_mean) / train_std
X_val   = (X_val   - train_mean) / train_std
X_test  = (X_test  - train_mean) / train_std

# ── Clip extreme values (EDA found artefacts up to ±13 mV on V2/V4/V5) ──
# After Z-normalisation, clip to ±10σ to prevent outlier signals from
# distorting the loss. This preserves >99.99% of data unchanged.
CLIP_STD = 10.0
X_train = np.clip(X_train, -CLIP_STD, CLIP_STD)
X_val   = np.clip(X_val,   -CLIP_STD, CLIP_STD)
X_test  = np.clip(X_test,  -CLIP_STD, CLIP_STD)

# Save normalisation params for deployment later
np.savez("normalisation_params.npz", mean=train_mean, std=train_std)
print("Normalisation + clipping applied. Stats saved to normalisation_params.npz")
print(f"Train range: [{X_train.min():.2f}, {X_train.max():.2f}]")

In [ ]:
# ── 3.3  Compute class weights (address imbalance) ─────────────────
# For multi-label BCE, we weight each class inversely proportional
# to its frequency, so rare conditions get more attention.

pos_counts = y_train.sum(axis=0)
neg_counts = len(y_train) - pos_counts
class_weights_array = neg_counts / (pos_counts + 1e-7)

print("Class weights (neg/pos ratio):")
for i, sc in enumerate(SUPERCLASSES):
    print(f"  {sc:5s}: {class_weights_array[i]:.2f}")

## 4. Model 1 — 1D Convolutional Neural Network <a id='cnn'></a>

**Architecture rationale:** 1D-CNNs are highly effective for ECG classification because they can learn local morphological features (QRS complex width, ST-segment elevation, T-wave inversions) through convolutional filters that slide along the temporal axis. Multiple layers of convolutions capture increasingly abstract patterns — from individual wave shapes to complex arrhythmia signatures.

Our architecture uses:
- **4 convolutional blocks** with increasing filter counts (32 → 64 → 128 → 256)
- **Batch normalisation** after each conv layer for training stability
- **MaxPooling** to reduce temporal resolution and build translation invariance
- **Dropout** for regularisation
- **Global Average Pooling** before the classifier head (more robust than Flatten, also required for Grad-CAM)

In [ ]:
# ── 4.1  Build the 1D-CNN ───────────────────────────────────────────

def build_cnn(input_shape, num_classes, dropout_rate=0.3):
    """
    1D-CNN for multi-label ECG classification.

    Architecture:
        Input (1000, 12)
          -> [Conv1D -> BN -> ReLU -> MaxPool -> Dropout] x 4 blocks
          -> GlobalAveragePooling1D
          -> Dense(128) -> Dropout -> Dense(num_classes, sigmoid)
    """
    inputs = keras.Input(shape=input_shape, name="ecg_input")

    # ── Block 1 ──
    x = layers.Conv1D(32, kernel_size=7, padding="same", name="conv1")(inputs)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(pool_size=2, name="pool1")(x)
    x = layers.Dropout(dropout_rate)(x)

    # ── Block 2 ──
    x = layers.Conv1D(64, kernel_size=5, padding="same", name="conv2")(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(pool_size=2, name="pool2")(x)
    x = layers.Dropout(dropout_rate)(x)

    # ── Block 3 ──
    x = layers.Conv1D(128, kernel_size=5, padding="same", name="conv3")(x)
    x = layers.BatchNormalization(name="bn3")(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(pool_size=2, name="pool3")(x)
    x = layers.Dropout(dropout_rate)(x)

    # ── Block 4 (last conv — target for Grad-CAM) ──
    x = layers.Conv1D(256, kernel_size=3, padding="same", name="conv4_gradcam")(x)
    x = layers.BatchNormalization(name="bn4")(x)
    x = layers.Activation("relu", name="conv4_relu")(x)
    x = layers.MaxPooling1D(pool_size=2, name="pool4")(x)
    x = layers.Dropout(dropout_rate)(x)

    # ── Classifier head ──
    x = layers.GlobalAveragePooling1D(name="gap")(x)
    x = layers.Dense(128, activation="relu", name="fc1")(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="sigmoid", name="predictions")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="ECG_1D_CNN")
    return model


INPUT_SHAPE = (X_train.shape[1], X_train.shape[2])  # (1000, 12)

cnn_model = build_cnn(INPUT_SHAPE, NUM_CLASSES)
cnn_model.summary()

## 5. Model 2 — LSTM Recurrent Network <a id='lstm'></a>

**Architecture rationale:** LSTMs capture long-range temporal dependencies in sequential data. For ECG signals, this means the model can learn relationships between events separated in time — for example, linking the morphology of the P-wave to the subsequent QRS complex, or detecting sustained ST-segment changes across multiple heartbeats.

Our architecture uses:
- **A 1D Conv front-end** (feature extraction + downsampling) to reduce sequence length before the LSTM — this makes training significantly faster and helps the LSTM focus on higher-level patterns
- **Bidirectional LSTM** so the model reads the ECG both forward and backward in time
- **Two stacked LSTM layers** for hierarchical temporal representation
- **Dropout** on both input and recurrent connections

In [ ]:
# ── 5.1  Build the LSTM ─────────────────────────────────────────────

def build_lstm(input_shape, num_classes, dropout_rate=0.3):
    """
    Bidirectional LSTM with a convolutional front-end for ECG classification.

    Architecture:
        Input (1000, 12)
          -> Conv1D(64, k=7) -> BN -> ReLU -> MaxPool(4)
          -> Bidirectional(LSTM(128, return_sequences)) -> Dropout
          -> Bidirectional(LSTM(64)) -> Dropout
          -> Dense(128) -> Dropout -> Dense(num_classes, sigmoid)
    """
    inputs = keras.Input(shape=input_shape, name="ecg_input")

    # ── Conv front-end: extract local features + downsample ──
    x = layers.Conv1D(64, kernel_size=7, padding="same", name="lstm_conv1")(inputs)
    x = layers.BatchNormalization(name="lstm_bn1")(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(pool_size=4, name="lstm_pool1")(x)  # 1000 -> 250
    x = layers.Dropout(dropout_rate)(x)

    # ── Recurrent layers ──
    x = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, dropout=dropout_rate),
        name="bi_lstm1"
    )(x)

    x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=False, dropout=dropout_rate),
        name="bi_lstm2"
    )(x)

    # ── Classifier head ──
    x = layers.Dense(128, activation="relu", name="lstm_fc1")(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="sigmoid", name="predictions")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="ECG_BiLSTM")
    return model


lstm_model = build_lstm(INPUT_SHAPE, NUM_CLASSES)
lstm_model.summary()

## 6. Training with MLflow Tracking <a id='training'></a>

We use [MLflow](https://mlflow.org/) to log training metrics, hyperparameters, and artifacts. This allows interactive comparison of experiments via the MLflow UI.

**Training details:**
- **Loss:** Binary cross-entropy (standard for multi-label classification with sigmoid outputs)
- **Optimiser:** Adam with initial LR of 1e-3
- **Callbacks:** ReduceLROnPlateau, EarlyStopping, ModelCheckpoint, MLflow autologging

After training, launch the MLflow UI with:
```bash
mlflow ui
```

In [ ]:
# ── 6.1  MLflow setup ───────────────────────────────────────────────
import mlflow
import mlflow.keras

# ── Force a local file-based tracking store (avoids the SQLite registry issue) ──
mlflow.set_tracking_uri("file:./mlruns")

# Set the experiment name (creates it if it doesn't exist)
EXPERIMENT_NAME = "ECG_Classification_DAT255"
mlflow.set_experiment(EXPERIMENT_NAME)

# Enable Keras autologging — this automatically captures:
# metrics per epoch, model architecture, optimizer config, and the trained model.
mlflow.keras.autolog()

print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print("After training, run 'mlflow ui' to launch the dashboard.")

In [ ]:
# ── 6.2  Training utility function ──────────────────────────────────

def train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    experiment_name,
    epochs=50,
    batch_size=64,
    learning_rate=1e-3,
):
    """
    Compile, train, and return a model with full MLflow tracking.
    """
    # ── Compile ──
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.AUC(name="auc", multi_label=True),
            keras.metrics.Recall(name="recall"),
            keras.metrics.Precision(name="precision"),
        ],
    )

    # ── Callbacks ──
    checkpoint_path = f"best_{experiment_name}.keras"

    cb_list = [
        callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1,
        ),
        callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=10,
            restore_best_weights=True,
            verbose=1,
        ),
    ]

    # ── MLflow run ──
    with mlflow.start_run(run_name=experiment_name):
        # Log hyperparameters
        mlflow.log_params({
            "model_name": model.name,
            "epochs": epochs,
            "batch_size": batch_size,
            "learning_rate": learning_rate,
            "dropout": 0.3,
            "optimizer": "Adam",
            "loss": "binary_crossentropy",
            "num_params": model.count_params(),
            "input_shape": str(INPUT_SHAPE),
            "sampling_rate": SAMPLING_RATE,
        })

        # Train (autolog captures per-epoch metrics automatically)
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=cb_list,
            verbose=1,
        )

        # Log the best model as an artifact
        mlflow.log_artifact(checkpoint_path)

        # Log final validation metrics explicitly
        best_epoch = np.argmax(history.history["val_auc"])
        mlflow.log_metrics({
            "best_val_auc": history.history["val_auc"][best_epoch],
            "best_val_loss": history.history["val_loss"][best_epoch],
            "best_val_accuracy": history.history["val_accuracy"][best_epoch],
            "best_epoch": best_epoch + 1,
        })

    # ── Load best weights ──
    model = keras.saving.load_model(checkpoint_path)
    print(f"\nLoaded best model from {checkpoint_path}")

    return model, history

In [ ]:
# ── 6.3  Train the 1D-CNN ──────────────────────────────────────────
print("=" * 60)
print("TRAINING: 1D-CNN")
print("=" * 60)

cnn_model, cnn_history = train_model(
    model=cnn_model,
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    experiment_name="1D_CNN",
    epochs=50,
    batch_size=64,
    learning_rate=1e-3,
)

In [ ]:
# ── 6.4  Train the BiLSTM ──────────────────────────────────────────
print("=" * 60)
print("TRAINING: Bidirectional LSTM")
print("=" * 60)

lstm_model, lstm_history = train_model(
    model=lstm_model,
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    experiment_name="BiLSTM",
    epochs=50,
    batch_size=64,
    learning_rate=1e-3,
)

In [ ]:
# ── 6.5  Plot training curves ──────────────────────────────────────

def plot_training_curves(histories, names, metrics=("loss", "accuracy", "auc")):
    """Plot training & validation curves for multiple models side by side."""
    n_metrics = len(metrics)
    fig, axes = plt.subplots(1, n_metrics, figsize=(6 * n_metrics, 5))
    colors = ["#2980b9", "#e74c3c"]

    for col, metric in enumerate(metrics):
        ax = axes[col]
        for i, (history, name) in enumerate(zip(histories, names)):
            ax.plot(history.history[metric], label=f"{name} train",
                    color=colors[i], linestyle="-")
            ax.plot(history.history[f"val_{metric}"], label=f"{name} val",
                    color=colors[i], linestyle="--")
        ax.set_title(metric.upper())
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()


plot_training_curves(
    [cnn_history, lstm_history],
    ["1D-CNN", "BiLSTM"],
)

## 7. Evaluation & Comparison <a id='evaluation'></a>

We evaluate on the **held-out test set** (fold 10) using:
- **Per-class Precision, Recall, F1-score**
- **Macro-averaged F1** (treats all classes equally — important given imbalance)
- **Subset accuracy** (exact match — strict)
- **Sample-based accuracy** (Jaccard-style per sample)
- **ROC-AUC** (macro, per-class)

In [ ]:
# ── 7.1  Evaluation utility ────────────────────────────────────────

def evaluate_model(model, X_test, y_test, model_name, threshold=0.5):
    """Full evaluation with classification report, AUC, and confusion matrices."""
    # Predict probabilities
    y_prob = model.predict(X_test, batch_size=128)
    y_pred = (y_prob >= threshold).astype(np.float32)

    # ── Classification report ──
    print(f"\n{'='*60}")
    print(f"  Evaluation: {model_name}")
    print(f"{'='*60}")
    print(classification_report(
        y_test, y_pred,
        target_names=SUPERCLASSES,
        digits=4,
        zero_division=0,
    ))

    # ── Key metrics ──
    f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
    f1_micro = f1_score(y_test, y_pred, average="micro", zero_division=0)
    try:
        auc_macro = roc_auc_score(y_test, y_prob, average="macro")
    except ValueError:
        auc_macro = float("nan")

    # Subset accuracy (exact match for all 5 labels)
    subset_acc = accuracy_score(y_test, y_pred)

    # Sample-based Jaccard accuracy
    sample_acc = np.mean([
        np.sum(y_test[i] * y_pred[i]) /
        max(np.sum(y_test[i] + y_pred[i] - y_test[i] * y_pred[i]), 1)
        for i in range(len(y_test))
    ])

    print(f"  F1 (macro)       : {f1_macro:.4f}")
    print(f"  F1 (micro)       : {f1_micro:.4f}")
    print(f"  ROC-AUC (macro)  : {auc_macro:.4f}")
    print(f"  Subset accuracy  : {subset_acc:.4f}")
    print(f"  Sample accuracy  : {sample_acc:.4f}")

    return {
        "name": model_name,
        "y_prob": y_prob,
        "y_pred": y_pred,
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "auc_macro": auc_macro,
        "subset_acc": subset_acc,
        "sample_acc": sample_acc,
    }

In [ ]:
# ── 7.2  Evaluate both models ─────────────────────────────────────
cnn_results  = evaluate_model(cnn_model, X_test, y_test, "1D-CNN")
lstm_results = evaluate_model(lstm_model, X_test, y_test, "BiLSTM")

In [ ]:
# ── 7.3  Side-by-side comparison table ─────────────────────────────
comparison = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ("y_prob", "y_pred")}
    for r in [cnn_results, lstm_results]
]).set_index("name")

print("\n" + "="*60)
print("  MODEL COMPARISON")
print("="*60)
print(comparison.to_string(float_format="{:.4f}".format))

In [ ]:
# ── 7.4  Per-class confusion matrices ──────────────────────────────

def plot_multilabel_confusion(y_true, y_pred, class_names, title):
    """Plot per-class binary confusion matrices."""
    fig, axes = plt.subplots(1, len(class_names), figsize=(4 * len(class_names), 4))
    cms = multilabel_confusion_matrix(y_true, y_pred)
    for i, (cm, name, ax) in enumerate(zip(cms, class_names, axes)):
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                    xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"])
        ax.set_title(name)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(f"confusion_{title.replace(' ', '_').lower()}.png", dpi=150)
    plt.show()


plot_multilabel_confusion(y_test, cnn_results["y_pred"], SUPERCLASSES, "1D-CNN")
plot_multilabel_confusion(y_test, lstm_results["y_pred"], SUPERCLASSES, "BiLSTM")

In [ ]:
# ── 7.5  ROC curves per class ──────────────────────────────────────
from sklearn.metrics import roc_curve, auc

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, results, name in zip(axes, [cnn_results, lstm_results],
                              ["1D-CNN", "BiLSTM"]):
    for i, sc in enumerate(SUPERCLASSES):
        fpr, tpr, _ = roc_curve(y_test[:, i], results["y_prob"][:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"{sc} (AUC={roc_auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_title(f"ROC Curves — {name}")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Grad-CAM Interpretability <a id='gradcam'></a>

**Grad-CAM** (Gradient-weighted Class Activation Mapping) highlights which temporal regions of the input ECG were most important for the model's prediction. For the 1D-CNN, we compute gradients of the predicted class score with respect to the last convolutional layer's activations, then weight the activation maps by the average gradient per filter.

This is clinically meaningful: a cardiologist can verify whether the model is attending to the correct ECG features (e.g., ST-segment for MI, wide QRS for CD).

In [ ]:
# ── 8.1  Grad-CAM implementation for 1D-CNN ───────────────────────
import tensorflow as tf  # needed for GradientTape


def compute_gradcam_1d(model, input_sample, class_idx,
                       last_conv_name="conv4_gradcam"):
    """
    Compute 1D Grad-CAM heatmap for a given class.

    Parameters
    ----------
    model : keras.Model
    input_sample : np.ndarray, shape (1, timesteps, channels)
    class_idx : int, index into the 5 superclasses
    last_conv_name : str, name of the last Conv1D layer

    Returns
    -------
    heatmap : np.ndarray, shape (timesteps,) — normalised to [0, 1]
    """
    # Build a sub-model that outputs both the conv activations and predictions
    grad_model = keras.Model(
        inputs=model.input,
        outputs=[
            model.get_layer(last_conv_name).output,
            model.output,
        ],
    )

    input_tensor = tf.cast(input_sample, tf.float32)

    with tf.GradientTape() as tape:
        conv_output, predictions = grad_model(input_tensor)
        class_score = predictions[:, class_idx]

    # Gradients of the class score w.r.t. the conv activations
    grads = tape.gradient(class_score, conv_output)  # (1, T', filters)

    # Global average pool over the temporal dimension -> weight per filter
    weights = tf.reduce_mean(grads, axis=1)  # (1, filters)

    # Weighted combination of activation maps
    cam = tf.reduce_sum(
        conv_output * weights[:, tf.newaxis, :], axis=-1
    )  # (1, T')
    cam = tf.nn.relu(cam).numpy().squeeze()  # only positive contributions

    # Upsample to original input length via interpolation
    original_length = input_sample.shape[1]
    cam_upsampled = np.interp(
        np.linspace(0, len(cam) - 1, original_length),
        np.arange(len(cam)),
        cam,
    )

    # Normalise to [0, 1]
    if cam_upsampled.max() > 0:
        cam_upsampled = cam_upsampled / cam_upsampled.max()

    return cam_upsampled

In [ ]:
# ── 8.2  Visualise Grad-CAM on test samples ───────────────────────

def plot_gradcam(model, X_test, y_test, sample_indices, lead_idx=1):
    """
    Plot ECG Lead II with Grad-CAM overlay for each predicted class.
    """
    time = np.arange(X_test.shape[1]) / SAMPLING_RATE

    for idx in sample_indices:
        sample = X_test[idx:idx+1]
        y_prob = model.predict(sample, verbose=0).squeeze()
        y_true = y_test[idx]

        # Find classes with probability > 0.5 (or true classes)
        active_classes = np.where(y_prob >= 0.5)[0]
        if len(active_classes) == 0:
            active_classes = np.where(y_true == 1)[0]

        n_plots = len(active_classes)
        fig, axes = plt.subplots(n_plots, 1, figsize=(14, 3 * n_plots),
                                 sharex=True)
        if n_plots == 1:
            axes = [axes]

        for ax, cls_idx in zip(axes, active_classes):
            heatmap = compute_gradcam_1d(model, sample, cls_idx)

            # Plot ECG signal
            ax.plot(time, sample[0, :, lead_idx], color="#2c3e50",
                    linewidth=0.8)

            # Overlay heatmap as colored background
            ax.imshow(
                heatmap[np.newaxis, :],
                cmap="Reds",
                alpha=0.4,
                vmin=0, vmax=1,
                aspect="auto",
                extent=[time[0], time[-1], *ax.get_ylim()],
                origin="lower"
            )

            true_label = "TRUE" if y_true[cls_idx] == 1 else "FALSE"
            ax.set_title(
                f"{SUPERCLASSES[cls_idx]} — prob: {y_prob[cls_idx]:.3f} | "
                f"ground truth: {true_label}",
                fontsize=11,
            )
            ax.set_ylabel(f"Lead {LEAD_NAMES[lead_idx]} (norm.)")

        axes[-1].set_xlabel("Time (s)")
        fig.suptitle(f"Grad-CAM — Sample {idx}", fontsize=13)
        plt.tight_layout()
        plt.savefig(f"gradcam_sample_{idx}.png", dpi=150, bbox_inches="tight")
        plt.show()


# Pick one test sample per class for visualisation
example_indices = []
for cls_idx in range(NUM_CLASSES):
    candidates = np.where(y_test[:, cls_idx] == 1)[0]
    if len(candidates) > 0:
        example_indices.append(candidates[0])

print(f"Visualising Grad-CAM for samples: {example_indices}")
plot_gradcam(cnn_model, X_test, y_test, example_indices)

## 9. Export for HuggingFace & Gradio Deployment <a id='deployment'></a>

We save the best model in Keras format and prepare:
1. **HuggingFace Hub** — push the model + normalisation params
2. **Gradio app** — REST API for inference

In [ ]:
# ── 9.1  Save both models ──────────────────────────────────────────
cnn_model.save("ecg_cnn_final.keras")
lstm_model.save("ecg_lstm_final.keras")
print("Models saved: ecg_cnn_final.keras, ecg_lstm_final.keras")

In [ ]:
from huggingface_hub import HfApi, upload_folder, login
# ── 9.2  Push to HuggingFace Hub ───────────────────────────────────
# First log in:  huggingface-cli login
import shutil

HF_REPO = "Steenslid/ecg-ptbxl-classification"  # <-- Change this!

# Prepare upload directory
hf_dir = "hf_upload"
os.makedirs(hf_dir, exist_ok=True)
shutil.copy("ecg_cnn_final.keras", hf_dir)
shutil.copy("ecg_lstm_final.keras", hf_dir)
shutil.copy("normalisation_params.npz", hf_dir)

# Write a model card
model_card = """---
tags:
  - ecg
  - cardiovascular
  - multi-label-classification
  - keras
  - ptb-xl
datasets:
  - ptb-xl
---

# ECG Cardiovascular Disease Classification

Multi-label classification of 5 cardiovascular conditions (NORM, MI, STTC, CD, HYP)
from 12-lead ECG recordings using the PTB-XL dataset.

## Models
- `ecg_cnn_final.keras` — 1D Convolutional Neural Network
- `ecg_lstm_final.keras` — Bidirectional LSTM

## Usage
```python
import keras
import numpy as np

model = keras.saving.load_model("ecg_cnn_final.keras")
params = np.load("normalisation_params.npz")
# Normalise your (1000, 12) ECG input:
x = (x - params["mean"]) / params["std"]
probs = model.predict(x[np.newaxis])
```

**Authors:** Edvard Vindenes Steenslid & Morten Kvamme — DAT255
"""

with open(os.path.join(hf_dir, "README.md"), "w") as f:
    f.write(model_card)

# ── Uncomment to actually push ──
api = HfApi()
login()
upload_folder(folder_path="hf_upload", repo_id="Steenslid/ecg-ptbxl-classification", repo_type="model")
print(f"Uploaded to https://huggingface.co/{HF_REPO}")

print(f"HuggingFace upload prepared in ./{hf_dir}/")
print("Set HF_REPO to your username, uncomment the upload lines, and run.")

In [ ]:
# ── 9.3  Gradio REST API (inline test) ─────────────────────────────
import gradio as gr

# Load model and normalisation params
inference_model = keras.saving.load_model("ecg_cnn_final.keras")
norm_params = np.load("normalisation_params.npz")
norm_mean = norm_params["mean"].squeeze()
norm_std  = norm_params["std"].squeeze()


def predict_ecg(file):
    """
    Accepts a .npy file with shape (1000, 12) — a single 12-lead ECG.
    Returns predicted probabilities for each superclass.
    """
    try:
        ecg = np.load(file.name).astype(np.float32)
        if ecg.shape != (1000, 12):
            return {"error": f"Expected shape (1000, 12), got {ecg.shape}."}

        ecg = (ecg - norm_mean) / norm_std
        probs = inference_model.predict(ecg[np.newaxis], verbose=0).squeeze()

        return {sc: float(f"{p:.4f}") for sc, p in zip(SUPERCLASSES, probs)}
    except Exception as e:
        return {"error": str(e)}


demo = gr.Interface(
    fn=predict_ecg,
    inputs=gr.File(label="Upload ECG (.npy, shape 1000x12)"),
    outputs=gr.JSON(label="Predicted Probabilities"),
    title="ECG Cardiovascular Disease Classifier",
    description=(
        "Upload a 12-lead ECG recording (100 Hz, 10 seconds) as a NumPy .npy file. "
        "Predicts probabilities for: NORM, MI, STTC, CD, HYP."
    ),
)

# Uncomment to launch:
# demo.launch(share=True)
print("Gradio app defined. Uncomment demo.launch() to start.")
print("REST API will be at: http://localhost:7860/api/predict")

---
## Summary

| Component | Details |
|---|---|
| **Data** | PTB-XL, 100 Hz, 12-lead, 5 diagnostic superclasses |
| **Preprocessing** | Z-score normalisation (per-channel, train-fitted) |
| **Model 1** | 1D-CNN: 4 conv blocks → GAP → Dense (sigmoid) |
| **Model 2** | BiLSTM: conv front-end → 2x Bidirectional LSTM → Dense (sigmoid) |
| **Training** | Adam, BCE loss, EarlyStopping, ReduceLR, MLflow autologging |
| **Evaluation** | F1 (macro/micro), ROC-AUC, per-class confusion matrices |
| **Interpretability** | Grad-CAM on last conv layer of 1D-CNN |
| **Deployment** | HuggingFace Hub + Gradio REST API |
| **Experiment tracking** | MLflow — `mlflow ui` |

### Next Steps
- Hyperparameter search (kernel sizes, depth, learning rate schedule)
- Data augmentation (time shift, Gaussian noise, lead dropout)
- Try 500 Hz signals for higher temporal resolution
- Per-class threshold tuning to optimise F1
- Write the DAT255 project report using the template